In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# Configurar matplotlib
plt.style.use('default')
%matplotlib inline

In [ ]:
# Cargar archivos de la red bipartita
nodos = pd.read_csv('nodos_bipartita.csv')
aristas = pd.read_csv('aristas_bipartita.csv')

# Separar por tipo
autores = nodos[nodos['tipo'] == 'autor']
videos = nodos[nodos['tipo'] == 'video']

print(f"Autores: {len(autores)}")
print(f"Videos: {len(videos)}")
print(f"Aristas (autor-video): {len(aristas)}")

# Fase 5: Proyecciones de la Red\n\n## 5.1 Proyección Autor-Autor

In [ ]:
# Construcción de la proyección autor-autor
# Dos autores están conectados si comentaron en al menos un mismo video
# El peso es el número de videos compartidos

# Crear diccionario: video -> lista de autores
video_to_autores = defaultdict(list)
for _, row in aristas.iterrows():
    video_to_autores[row['video']].append(row['autor'])

# Crear aristas de autor-autor
autor_autor_edges = defaultdict(int)
for video, autores_lista in video_to_autores.items():
    # Para cada par de autores en este video, incrementar contador
    for i in range(len(autores_lista)):
        for j in range(i + 1, len(autores_lista)):
            a1, a2 = sorted([autores_lista[i], autores_lista[j]])
            autor_autor_edges[(a1, a2)] += 1

# Convertir a DataFrame
df_aa = pd.DataFrame([
    {'author_1': k[0], 'author_2': k[1], 'weight': v}
    for k, v in autor_autor_edges.items()
])

print(f"Aristas autor-autor: {len(df_aa)}")
print(f"\nPrimeras 10 aristas:")
print(df_aa.head(10))

In [ ]:
# Validaciones para proyección autor-autor
nodos_aa = set(df_aa['author_1'].unique()) | set(df_aa['author_2'].unique())
n_nodos_aa = len(nodos_aa)
n_aristas_aa = len(df_aa)
n_aislados_aa = len(autores) - n_nodos_aa

print("PROYECCIÓN AUTOR-AUTOR")
print(f"Número de nodos: {n_nodos_aa}")
print(f"Número de aristas: {n_aristas_aa}")
print(f"Nodos aislados: {n_aislados_aa}")
print(f"Peso mínimo: {df_aa['weight'].min()}")
print(f"Peso máximo: {df_aa['weight'].max()}")
print(f"Peso promedio: {df_aa['weight'].mean():.2f}")

In [ ]:
# Tabla de top 10 aristas autor-autor con nombres
df_aa_top = df_aa.nlargest(10, 'weight').copy()

# Fusionar con nombres de autores
author_names = autores[['id', 'nombre']].set_index('id')
df_aa_top['author_name_1'] = df_aa_top['author_1'].map(author_names['nombre'])
df_aa_top['author_name_2'] = df_aa_top['author_2'].map(author_names['nombre'])

print("Top 10 aristas autor-autor (por peso - videos compartidos):")
print(df_aa_top[['author_1', 'author_name_1', 'author_2', 'author_name_2', 'weight']])

In [ ]:
# Guardar aristas autor-autor
df_aa.to_csv('aristas_autor_autor.csv', index=False)

# Crear tabla de nodos autor-autor
nodos_aa_table = autores[['id', 'nombre', 'n_comentarios', 'n_videos']].copy()
nodos_aa_table.columns = ['author_channel_id', 'author_name', 'n_comentarios', 'n_videos']

# Agregar grado en la red autor-autor
grado_dict = defaultdict(int)
for _, row in df_aa.iterrows():
    grado_dict[row['author_1']] += 1
    grado_dict[row['author_2']] += 1

nodos_aa_table['grado_aa'] = nodos_aa_table['author_channel_id'].map(lambda x: grado_dict.get(x, 0))
nodos_aa_table.to_csv('nodos_autor_autor.csv', index=False)

print("Archivos guardados: aristas_autor_autor.csv, nodos_autor_autor.csv")

## 5.2 Proyección Video-Video

In [ ]:
# Construcción de la proyección video-video
# Dos videos están conectados si comparten al menos un autor
# El peso es el número de autores únicos compartidos

# Crear diccionario: autor -> lista de videos
autor_to_videos = defaultdict(list)
for _, row in aristas.iterrows():
    autor_to_videos[row['autor']].append(row['video'])

# Crear aristas de video-video
video_video_edges = defaultdict(int)
for autor, videos_lista in autor_to_videos.items():
    # Para cada par de videos comentados por este autor, incrementar contador
    for i in range(len(videos_lista)):
        for j in range(i + 1, len(videos_lista)):
            v1, v2 = sorted([videos_lista[i], videos_lista[j]])
            video_video_edges[(v1, v2)] += 1

# Convertir a DataFrame
df_vv = pd.DataFrame([
    {'video_1': k[0], 'video_2': k[1], 'weight': v}
    for k, v in video_video_edges.items()
])

print(f"Aristas video-video: {len(df_vv)}")
print(f"\nPrimeras 10 aristas:")
print(df_vv.head(10))

In [ ]:
# Validaciones para proyección video-video
nodos_vv = set(df_vv['video_1'].unique()) | set(df_vv['video_2'].unique())
n_nodos_vv = len(nodos_vv)
n_aristas_vv = len(df_vv)
n_aislados_vv = len(videos) - n_nodos_vv

print("PROYECCIÓN VIDEO-VIDEO")
print(f"Número de nodos: {n_nodos_vv}")
print(f"Número de aristas: {n_aristas_vv}")
print(f"Nodos aislados: {n_aislados_vv}")
print(f"Peso mínimo: {df_vv['weight'].min()}")
print(f"Peso máximo: {df_vv['weight'].max()}")
print(f"Peso promedio: {df_vv['weight'].mean():.2f}")

In [ ]:
# Tabla de top 10 aristas video-video con títulos
df_vv_top = df_vv.nlargest(10, 'weight').copy()

# Fusionar con títulos de videos
video_info = videos[['id', 'titulo', 'canal']].set_index('id')
df_vv_top['title_1'] = df_vv_top['video_1'].map(video_info['titulo'])
df_vv_top['channel_1'] = df_vv_top['video_1'].map(video_info['canal'])
df_vv_top['title_2'] = df_vv_top['video_2'].map(video_info['titulo'])
df_vv_top['channel_2'] = df_vv_top['video_2'].map(video_info['canal'])

print("Top 10 aristas video-video (por peso - autores compartidos):")
print(df_vv_top[['video_1', 'title_1', 'video_2', 'title_2', 'weight']])

In [ ]:
# Guardar aristas video-video
df_vv.to_csv('aristas_video_video.csv', index=False)

# Crear tabla de nodos video-video
nodos_vv_table = videos[['id', 'titulo', 'canal', 'categoria']].copy()
nodos_vv_table.columns = ['video_id', 'title', 'channel_name', 'category']

# Agregar grado en la red video-video
grado_dict = defaultdict(int)
for _, row in df_vv.iterrows():
    grado_dict[row['video_1']] += 1
    grado_dict[row['video_2']] += 1

nodos_vv_table['grado_vv'] = nodos_vv_table['video_id'].map(lambda x: grado_dict.get(x, 0))
nodos_vv_table.to_csv('nodos_video_video.csv', index=False)

print("Archivos guardados: aristas_video_video.csv, nodos_video_video.csv")

## 5.4 Visualizaciones\n\n### Visualización Autor-Autor

In [ ]:
# Visualización autor-autor usando matplotlib directamente
# Usar spring layout manual (algoritmo de fuerzas simplificado)
import matplotlib.patches as mpatches

# Preparar datos para visualización
df_aa_viz = df_aa.copy()
nodos_set = list(nodos_aa)
n_nodos = len(nodos_set)
nodo_to_idx = {n: i for i, n in enumerate(nodos_set)}

# Posiciones iniciales (círculo)
np.random.seed(42)
angulos = np.linspace(0, 2*np.pi, n_nodos, endpoint=False)
posiciones = {n: (np.cos(angulos[i]), np.sin(angulos[i])) for i, n in enumerate(nodos_set)}

# Crear figura
fig, ax = plt.subplots(figsize=(14, 12))

# Dibujar aristas
for _, row in df_aa_viz.iterrows():
    x1, y1 = posiciones[row['author_1']]
    x2, y2 = posiciones[row['author_2']]
    w = row['weight']
    ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.3, linewidth=0.5 + 0.3*w)

# Dibujar nodos
node_xs = [posiciones[n][0] for n in nodos_set]
node_ys = [posiciones[n][1] for n in nodos_set]
node_sizes = [30 + grado_dict.get(n, 0)*20 for n in nodos_set]

ax.scatter(node_xs, node_ys, s=node_sizes, alpha=0.6, color='#3498db', edgecolors='#2c3e50', linewidth=1)

ax.set_title('Red Autor-Autor\n(Tamaño de nodo = grado en red)', fontsize=14, fontweight='bold')
ax.axis('off')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('graficos/05_red_autor_autor.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Visualización guardada: graficos/05_red_autor_autor.png")

In [ ]:
# Visualización video-video
df_vv_viz = df_vv.copy()
nodos_set_vv = list(nodos_vv)
n_nodos_vv = len(nodos_set_vv)
nodo_to_idx_vv = {n: i for i, n in enumerate(nodos_set_vv)}

# Posiciones iniciales (círculo)
np.random.seed(42)
angulos_vv = np.linspace(0, 2*np.pi, n_nodos_vv, endpoint=False)
posiciones_vv = {n: (np.cos(angulos_vv[i]), np.sin(angulos_vv[i])) for i, n in enumerate(nodos_set_vv)}

# Crear figura
fig, ax = plt.subplots(figsize=(14, 12))

# Dibujar aristas
grado_dict_vv = defaultdict(int)
for _, row in df_vv_viz.iterrows():
    x1, y1 = posiciones_vv[row['video_1']]
    x2, y2 = posiciones_vv[row['video_2']]
    w = row['weight']
    ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.3, linewidth=0.5 + 0.3*w)
    grado_dict_vv[row['video_1']] += 1
    grado_dict_vv[row['video_2']] += 1

# Dibujar nodos
node_xs_vv = [posiciones_vv[n][0] for n in nodos_set_vv]
node_ys_vv = [posiciones_vv[n][1] for n in nodos_set_vv]
node_sizes_vv = [50 + grado_dict_vv.get(n, 0)*30 for n in nodos_set_vv]

ax.scatter(node_xs_vv, node_ys_vv, s=node_sizes_vv, alpha=0.6, color='#e74c3c', edgecolors='#2c3e50', linewidth=1)

# Etiquetas para videos principales (grado alto)
for n in nodos_set_vv:
    if grado_dict_vv[n] >= 2:
        x, y = posiciones_vv[n]
        titulo = videos[videos['id'] == n]['titulo'].values
        if len(titulo) > 0:
            titulo_short = titulo[0][:20] + '...' if len(titulo[0]) > 20 else titulo[0]
            ax.text(x, y-0.15, titulo_short, fontsize=8, ha='center')

ax.set_title('Red Video-Video\n(Tamaño de nodo = grado en red)', fontsize=14, fontweight='bold')
ax.axis('off')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('graficos/06_red_video_video.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Visualización guardada: graficos/06_red_video_video.png")

## 5.3 Comparación de Proyecciones

In [ ]:
# Tabla comparativa
comparacion = pd.DataFrame({
    'Métrica': [
        'Nodos originales',
        'Nodos proyectados',
        'Nodos aislados',
        'Aristas',
        'Peso mín',
        'Peso máx',
        'Peso promedio'
    ],
    'Autor-Autor': [
        len(autores),
        n_nodos_aa,
        n_aislados_aa,
        n_aristas_aa,
        df_aa['weight'].min(),
        df_aa['weight'].max(),
        f"{df_aa['weight'].mean():.2f}"
    ],
    'Video-Video': [
        len(videos),
        n_nodos_vv,
        n_aislados_vv,
        n_aristas_vv,
        df_vv['weight'].min(),
        df_vv['weight'].max(),
        f"{df_vv['weight'].mean():.2f}"
    ]
})

print("COMPARACIÓN DE PROYECCIONES")
print(comparacion.to_string(index=False))

comparacion.to_csv('comparacion_proyecciones.csv', index=False)
print("\nArchivo guardado: comparacion_proyecciones.csv")